# E5 - Validacion holdout multiclase agrupado sagital

Este notebook valida el mejor modelo multiclase agrupado del notebook 09 sobre casos T1 de SPIDER no usados en el split interno de 100 casos.

Alcance:

- No entrena un modelo nuevo.
- No modifica arquitectura, mapping ni estrategia principal.
- Carga el checkpoint `E5_multiclass_unet2d_grouped_best.pt`.
- Usa el modelo binario mejorado solo como selector de slice cuando esta disponible.
- Reporta Dice/IoU macro sin fondo, metricas por clase, sanity checks, comparacion contra validacion interna y conclusion tecnica.

La salida es evidencia academica revisable. No emite diagnosticos ni recomendaciones clinicas.

In [1]:
# Dependencias para Google Colab.
# Ejecutar solo si SimpleITK o scikit-image no estan disponibles.
try:
    import SimpleITK  # noqa: F401
    import skimage  # noqa: F401
except Exception:
    %pip install -q SimpleITK scikit-image

In [2]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn as nn
from skimage.transform import resize
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 120)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("GPU disponible:", torch.cuda.is_available())
print("Dispositivo:", DEVICE)
if DEVICE.type == "cpu":
    print("ADVERTENCIA: el notebook puede correr en CPU, pero la inferencia sera mas lenta.")

GPU disponible: False
Dispositivo: cpu
ADVERTENCIA: el notebook puede correr en CPU, pero la inferencia sera mas lenta.


## Rutas y parametros

In [4]:
DATASET_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/SPIDER")
PREPROCESS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E4_preprocesamiento")
BINARY_IMPROVED_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_baseline_mejorado_binario")
MULTICLASS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_multiclase_agrupado")
MULTICLASS_HOLDOUT_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout")
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

CANDIDATES_CSV = PREPROCESS_ROOT / "E4_baseline_candidates_no_space.csv"
SELECTED_CASES_09_CSV = MULTICLASS_ROOT / "E5_multiclass_selected_cases.csv"
MAPPING_JSON = MULTICLASS_ROOT / "E5_multiclass_label_mapping.json"
MULTICLASS_MODEL_PATH = MULTICLASS_ROOT / "E5_multiclass_unet2d_grouped_best.pt"
INTERNAL_REPORT_JSON = MULTICLASS_ROOT / "E5_multiclass_validation_report.json"
BINARY_MODEL_PATH = BINARY_IMPROVED_ROOT / "E5_improved_unet2d_binary_best.pt"

MODALITY_FILTER = "t1"
N_HOLDOUT = 40
RANDOM_SEED = 42
CENTER_WINDOW_RADIUS = 3
INFERENCE_BATCH_SIZE = 4

MULTICLASS_HOLDOUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
DOCS_ROOT.mkdir(parents=True, exist_ok=True)

required_inputs = {
    "candidatos_e4": CANDIDATES_CSV,
    "casos_seleccionados_09": SELECTED_CASES_09_CSV,
    "mapping": MAPPING_JSON,
    "modelo_multiclase_best": MULTICLASS_MODEL_PATH,
    "reporte_interno_09": INTERNAL_REPORT_JSON,
}

missing_required = [name for name, path in required_inputs.items() if not path.exists()]
if missing_required:
    raise FileNotFoundError({
        name: str(required_inputs[name])
        for name in missing_required
    })

print("MULTICLASS_HOLDOUT_ROOT:", MULTICLASS_HOLDOUT_ROOT)
print("BINARY_MODEL_PATH existe:", BINARY_MODEL_PATH.exists())

MULTICLASS_HOLDOUT_ROOT: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout
BINARY_MODEL_PATH existe: True


## Reconstruccion de modelos y carga de checkpoints

In [5]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SimpleUNet2D(nn.Module):
    '''U-Net 2D binaria compatible con checkpoints con capa final `out`.'''

    def __init__(self, in_channels=1, out_channels=1, base_channels=16):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base_channels * 4, base_channels * 8)
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base_channels * 2, base_channels)
        self.out = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)


class SimpleUNet2DMulticlass(nn.Module):
    '''Misma U-Net 2D multiclase usada en el notebook 09.'''

    def __init__(self, in_channels=1, num_classes=4, base_channels=16):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base_channels * 4, base_channels * 8)
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base_channels * 2, base_channels)
        self.out_conv = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


def normalize_binary_state_dict_keys(state_dict):
    normalized = {}
    for key, value in state_dict.items():
        if key.startswith("out_conv."):
            key = key.replace("out_conv.", "out.", 1)
        normalized[key] = value
    return normalized

In [6]:
multiclass_checkpoint = torch.load(MULTICLASS_MODEL_PATH, map_location=DEVICE)

NUM_CLASSES = int(multiclass_checkpoint.get("num_classes", 4))
BASE_CHANNELS = int(multiclass_checkpoint.get("base_channels", 16))
TARGET_SIZE = tuple(multiclass_checkpoint.get("target_size", (256, 256)))
SAGITTAL_AXIS = int(multiclass_checkpoint.get("sagittal_axis", 2))
CHECKPOINT_SLICE_STRATEGY = str(multiclass_checkpoint.get("slice_strategy", "center_window_best_prediction"))

with open(MAPPING_JSON, "r", encoding="utf-8") as f:
    mapping_from_file = json.load(f)

mapping_from_checkpoint = multiclass_checkpoint.get("label_group_mapping", mapping_from_file)
LABEL_GROUP_MAPPING = {int(k): int(v) for k, v in mapping_from_checkpoint.items()}

model = SimpleUNet2DMulticlass(
    in_channels=1,
    num_classes=NUM_CLASSES,
    base_channels=BASE_CHANNELS,
).to(DEVICE)
model.load_state_dict(multiclass_checkpoint["model_state_dict"])
model.eval()

with torch.no_grad():
    dummy = torch.zeros((1, 1, TARGET_SIZE[0], TARGET_SIZE[1]), dtype=torch.float32, device=DEVICE)
    dummy_logits = model(dummy)

expected_shape = (1, NUM_CLASSES, TARGET_SIZE[0], TARGET_SIZE[1])
assert tuple(dummy_logits.shape) == expected_shape, f"Salida inesperada: {tuple(dummy_logits.shape)} vs {expected_shape}"

print("Modelo multiclase cargado:", MULTICLASS_MODEL_PATH)
print("NUM_CLASSES:", NUM_CLASSES)
print("BASE_CHANNELS:", BASE_CHANNELS)
print("TARGET_SIZE:", TARGET_SIZE)
print("SAGITTAL_AXIS:", SAGITTAL_AXIS)
print("SLICE_STRATEGY checkpoint:", CHECKPOINT_SLICE_STRATEGY)
print("val_dice_macro_no_bg checkpoint:", multiclass_checkpoint.get("val_dice_macro_no_bg"))

Modelo multiclase cargado: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_agrupado/E5_multiclass_unet2d_grouped_best.pt
NUM_CLASSES: 4
BASE_CHANNELS: 16
TARGET_SIZE: (256, 256)
SAGITTAL_AXIS: 2
SLICE_STRATEGY checkpoint: center_window_best_prediction
val_dice_macro_no_bg checkpoint: 0.8613568743069967


In [7]:
binary_selector_model = None
slice_strategy_used = "central_slice_fallback"
binary_selector_status = {
    "available": False,
    "path": str(BINARY_MODEL_PATH),
    "reason": None,
}

if BINARY_MODEL_PATH.exists():
    try:
        binary_checkpoint = torch.load(BINARY_MODEL_PATH, map_location=DEVICE)
        binary_base_channels = int(binary_checkpoint.get("base_channels", 16))
        binary_selector_model = SimpleUNet2D(
            in_channels=1,
            out_channels=1,
            base_channels=binary_base_channels,
        ).to(DEVICE)
        state_dict = normalize_binary_state_dict_keys(binary_checkpoint["model_state_dict"])
        missing_keys, unexpected_keys = binary_selector_model.load_state_dict(state_dict, strict=False)
        if missing_keys or unexpected_keys:
            print("Advertencia carga binaria strict=False:", {"missing": missing_keys, "unexpected": unexpected_keys})
        binary_selector_model.eval()
        slice_strategy_used = "center_window_best_prediction"
        binary_selector_status.update({
            "available": True,
            "base_channels": binary_base_channels,
            "missing_keys": list(missing_keys),
            "unexpected_keys": list(unexpected_keys),
        })
    except Exception as exc:
        binary_selector_model = None
        binary_selector_status["reason"] = repr(exc)
else:
    binary_selector_status["reason"] = "No existe el checkpoint binario selector."

print("Selector binario disponible:", binary_selector_model is not None)
print("Estrategia de slice utilizada:", slice_strategy_used)
if binary_selector_status["reason"]:
    print("Motivo fallback:", binary_selector_status["reason"])

Selector binario disponible: True
Estrategia de slice utilizada: center_window_best_prediction


## Construccion del holdout

In [8]:
def infer_case_modality(row):
    text = " ".join(str(v).lower() for v in row.values)
    if "t2" in text:
        return "t2"
    if "t1" in text:
        return "t1"
    return "unknown"


def ensure_case_id(df):
    out = df.copy()
    if "case_id" not in out.columns:
        for column in ["file_name", "image_path", "source_image_path", "mask_path", "source_mask_path"]:
            if column in out.columns:
                out["case_id"] = out[column].apply(lambda value: Path(str(value)).stem)
                break
    if "case_id" not in out.columns:
        raise ValueError("No se pudo construir case_id.")
    out["case_id"] = out["case_id"].astype(str)
    return out


def candidate_case_id_from_row(row):
    if "case_id" in row and pd.notna(row["case_id"]):
        return str(row["case_id"])
    for column in ["file_name", "image_path", "source_image_path", "mask_path", "source_mask_path"]:
        if column in row and pd.notna(row[column]):
            return Path(str(row[column])).stem
    return None


def first_existing_path(paths):
    for path in paths:
        if path is not None and Path(path).exists():
            return Path(path)
    return None


def resolve_existing_dataset_path(row, kind):
    if kind not in {"image", "mask"}:
        raise ValueError("kind debe ser 'image' o 'mask'")

    case_id = candidate_case_id_from_row(row)
    columns = (
        ["image_path", "source_image_path", "image", "img_path"]
        if kind == "image"
        else ["mask_path", "source_mask_path", "mask", "seg_path"]
    )

    direct_candidates = []
    for column in columns:
        if column in row and pd.notna(row[column]):
            direct_candidates.append(Path(str(row[column])))

    base_candidates = []
    if case_id:
        if kind == "image":
            base_candidates.extend([
                DATASET_ROOT / "images" / "images" / f"{case_id}.mha",
                DATASET_ROOT / "images" / f"{case_id}.mha",
                DATASET_ROOT / f"{case_id}.mha",
            ])
        else:
            base_candidates.extend([
                DATASET_ROOT / "masks" / "masks" / f"{case_id}.mha",
                DATASET_ROOT / "masks" / f"{case_id}.mha",
                DATASET_ROOT / f"{case_id}.mha",
            ])

    found = first_existing_path(direct_candidates + base_candidates)
    if found is not None:
        return found

    if case_id and DATASET_ROOT.exists():
        matches = list(DATASET_ROOT.rglob(f"{case_id}.mha"))
        if matches:
            preferred = [path for path in matches if kind in str(path).lower()]
            return preferred[0] if preferred else matches[0]

    return None


def get_case_paths(row):
    image_path = resolve_existing_dataset_path(row, "image")
    mask_path = resolve_existing_dataset_path(row, "mask")
    if image_path is None or mask_path is None:
        raise FileNotFoundError(f"No se encontraron imagen/mascara para case_id={candidate_case_id_from_row(row)}")
    return image_path, mask_path

In [9]:
candidates_df = ensure_case_id(pd.read_csv(CANDIDATES_CSV))
selected_09_df = ensure_case_id(pd.read_csv(SELECTED_CASES_09_CSV))
used_case_ids_09 = set(selected_09_df["case_id"].astype(str))

if "modality" in candidates_df.columns:
    candidates_df["inferred_modality"] = candidates_df["modality"].astype(str).str.lower()
else:
    candidates_df["inferred_modality"] = candidates_df.apply(infer_case_modality, axis=1)

t1_candidates_df = candidates_df[candidates_df["inferred_modality"].eq(MODALITY_FILTER)].copy()

valid_rows = []
missing_rows = []
for _, row in tqdm(t1_candidates_df.iterrows(), total=len(t1_candidates_df), desc="Resolviendo archivos T1"):
    case_id = str(row["case_id"])
    try:
        image_path, mask_path = get_case_paths(row)
        item = row.to_dict()
        item["source_image_path"] = str(image_path)
        item["source_mask_path"] = str(mask_path)
        valid_rows.append(item)
    except Exception as exc:
        item = row.to_dict()
        item["missing_reason"] = repr(exc)
        missing_rows.append(item)

valid_t1_df = pd.DataFrame(valid_rows)
missing_files_df = pd.DataFrame(missing_rows)

holdout_pool_df = valid_t1_df[~valid_t1_df["case_id"].astype(str).isin(used_case_ids_09)].copy()
holdout_selected_df = (
    holdout_pool_df
    .sample(n=min(N_HOLDOUT, len(holdout_pool_df)), random_state=RANDOM_SEED)
    .sort_values("case_id")
    .reset_index(drop=True)
)

holdout_selected_csv_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_selected_cases.csv"
holdout_missing_csv_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_missing_files.csv"
holdout_selected_df.to_csv(holdout_selected_csv_path, index=False)
if len(missing_files_df) > 0:
    missing_files_df.to_csv(holdout_missing_csv_path, index=False)

print("Candidatos T1:", len(t1_candidates_df))
print("Casos usados notebook 09:", len(used_case_ids_09))
print("T1 validos con archivos:", len(valid_t1_df))
print("Disponibles para holdout:", len(holdout_pool_df))
print("Seleccionados holdout:", len(holdout_selected_df))
print("CSV holdout:", holdout_selected_csv_path)
if len(missing_files_df) > 0:
    print("CSV faltantes:", holdout_missing_csv_path)

if len(holdout_selected_df) == 0:
    raise RuntimeError("No hay casos holdout disponibles despues de excluir los 100 casos del notebook 09.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Resolviendo archivos T1:   0%|          | 0/196 [00:00<?, ?it/s]

Candidatos T1: 196
Casos usados notebook 09: 100
T1 validos con archivos: 196
Disponibles para holdout: 96
Seleccionados holdout: 40
CSV holdout: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout/E5_multiclass_holdout_selected_cases.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Preprocesamiento y dataset holdout

In [10]:
unmapped_label_events = []


def read_mha(path):
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f"No existe el archivo MHA: {path}")
    itk_image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(itk_image)
    return itk_image, array


def robust_percentile_normalize(image_array, p_low=1, p_high=99, eps=1e-8):
    image_float = image_array.astype(np.float32)
    low, high = np.percentile(image_float, [p_low, p_high])
    clipped = np.clip(image_float, low, high)
    if np.isclose(high, low):
        return np.zeros_like(clipped, dtype=np.float32)
    return ((clipped - low) / (high - low + eps)).astype(np.float32)


def take_slice(array, axis, index):
    return np.take(array, indices=int(index), axis=int(axis))


def resize_image_slice(image_slice, target_size=TARGET_SIZE):
    out = resize(
        image_slice,
        target_size,
        order=1,
        mode="constant",
        anti_aliasing=True,
        preserve_range=True,
    )
    return out.astype(np.float32)


def resize_mask_slice(mask_slice, target_size=TARGET_SIZE):
    out = resize(
        mask_slice,
        target_size,
        order=0,
        mode="constant",
        anti_aliasing=False,
        preserve_range=True,
    )
    return out.astype(np.int64)


def map_mask_to_grouped(mask, case_id=None):
    grouped = np.zeros_like(mask, dtype=np.int64)
    unique_labels = np.unique(mask)
    unmapped = []
    for label in unique_labels:
        label_int = int(label)
        if label_int in LABEL_GROUP_MAPPING:
            grouped[mask == label_int] = int(LABEL_GROUP_MAPPING[label_int])
        else:
            unmapped.append(label_int)
            grouped[mask == label_int] = 0
    if unmapped:
        unmapped_label_events.append({
            "case_id": str(case_id),
            "unmapped_labels": sorted(set(unmapped)),
        })
    return grouped.astype(np.int64)


def candidate_indices_for_center_window(n_slices, radius=CENTER_WINDOW_RADIUS):
    center = n_slices // 2
    start = max(0, center - radius)
    end = min(n_slices - 1, center + radius)
    return list(range(start, end + 1))


def select_slice_center_window_prediction(image_norm, axis=SAGITTAL_AXIS):
    n_slices = image_norm.shape[axis]
    if binary_selector_model is None:
        return n_slices // 2

    candidate_indices = candidate_indices_for_center_window(n_slices)
    tensors = []
    for idx in candidate_indices:
        image_slice = take_slice(image_norm, axis, idx)
        image_slice_resized = resize_image_slice(image_slice, TARGET_SIZE)
        tensors.append(torch.from_numpy(image_slice_resized).unsqueeze(0))

    batch = torch.stack(tensors, dim=0).to(DEVICE)
    with torch.no_grad():
        logits = binary_selector_model(batch)
        probs = torch.sigmoid(logits)
        foreground_scores = (probs >= 0.5).float().mean(dim=(1, 2, 3)).detach().cpu().numpy()

    best_local = int(np.argmax(foreground_scores))
    return int(candidate_indices[best_local])


def prepare_holdout_sample(row):
    case_id = str(row["case_id"])
    image_path = Path(row["source_image_path"])
    mask_path = Path(row["source_mask_path"])
    _, image = read_mha(image_path)
    _, mask = read_mha(mask_path)

    if tuple(image.shape) != tuple(mask.shape):
        raise ValueError(f"Shape incompatible para {case_id}: image={image.shape}, mask={mask.shape}")

    image_norm = robust_percentile_normalize(image)
    slice_index = select_slice_center_window_prediction(image_norm, axis=SAGITTAL_AXIS)

    image_slice = take_slice(image_norm, SAGITTAL_AXIS, slice_index)
    mask_slice_original = take_slice(mask, SAGITTAL_AXIS, slice_index)
    mask_slice_grouped = map_mask_to_grouped(mask_slice_original, case_id=case_id)

    image_resized = resize_image_slice(image_slice, TARGET_SIZE)
    mask_resized = resize_mask_slice(mask_slice_grouped, TARGET_SIZE)
    mask_unique = sorted(int(v) for v in np.unique(mask_resized))

    if min(mask_unique) < 0 or max(mask_unique) >= NUM_CLASSES:
        raise ValueError(f"Mask fuera de rango para {case_id}: {mask_unique}")

    return {
        "case_id": case_id,
        "image": image_resized.astype(np.float32),
        "mask": mask_resized.astype(np.int64),
        "slice_index": int(slice_index),
        "image_shape": tuple(int(v) for v in image.shape),
        "mask_unique_values": mask_unique,
        "gt_foreground_ratio": float(np.mean(mask_resized > 0)),
        "source_image_path": str(image_path),
        "source_mask_path": str(mask_path),
    }

In [11]:
samples = []
sample_errors = []
for _, row in tqdm(holdout_selected_df.iterrows(), total=len(holdout_selected_df), desc="Preparando holdout"):
    try:
        samples.append(prepare_holdout_sample(row))
    except Exception as exc:
        error_row = row.to_dict()
        error_row["error"] = repr(exc)
        sample_errors.append(error_row)

if sample_errors:
    sample_errors_df = pd.DataFrame(sample_errors)
    sample_errors_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_sample_errors.csv"
    sample_errors_df.to_csv(sample_errors_path, index=False)
    print("Casos omitidos por error de preparacion:", sample_errors_path)

if len(samples) == 0:
    raise RuntimeError("No quedaron muestras validas para evaluar.")

samples_summary_df = pd.DataFrame([
    {
        "case_id": sample["case_id"],
        "selected_slice_index": sample["slice_index"],
        "image_shape": str(sample["image_shape"]),
        "mask_unique_values": json.dumps(sample["mask_unique_values"]),
        "gt_foreground_ratio": sample["gt_foreground_ratio"],
        "source_image_path": sample["source_image_path"],
        "source_mask_path": sample["source_mask_path"],
    }
    for sample in samples
])
samples_summary_csv_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_samples_summary.csv"
samples_summary_df.to_csv(samples_summary_csv_path, index=False)

unmapped_labels_csv_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_unmapped_labels.csv"
if unmapped_label_events:
    pd.DataFrame(unmapped_label_events).to_csv(unmapped_labels_csv_path, index=False)

print("Muestras validas:", len(samples))
print("Resumen muestras:", samples_summary_csv_path)
if unmapped_label_events:
    print("Labels no mapeados:", unmapped_labels_csv_path)
display(samples_summary_df.head())

Preparando holdout:   0%|          | 0/40 [00:00<?, ?it/s]

Muestras validas: 40
Resumen muestras: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout/E5_multiclass_holdout_samples_summary.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,case_id,selected_slice_index,image_shape,mask_unique_values,gt_foreground_ratio,source_image_path,source_mask_path
0,101_t1,8,"(298, 320, 17)","[0, 1, 2, 3]",0.188095,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
1,116_t1,5,"(320, 320, 15)","[0, 1, 2, 3]",0.087296,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
2,117_t1,13,"(448, 448, 24)","[0, 1, 2, 3]",0.100555,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
3,12_t1,8,"(320, 320, 15)","[0, 1, 2, 3]",0.142044,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...
4,131_t1,12,"(463, 448, 24)","[0, 1, 2, 3]",0.139160,/content/drive/MyDrive/PFI_MVP/data/SPIDER/ima...,/content/drive/MyDrive/PFI_MVP/data/SPIDER/mas...


In [12]:
class HoldoutMulticlassDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = torch.from_numpy(sample["image"]).unsqueeze(0).float()
        mask = torch.from_numpy(sample["mask"]).long()
        return {
            "case_id": sample["case_id"],
            "image": image,
            "mask": mask,
            "slice_index": sample["slice_index"],
            "source_image_path": sample["source_image_path"],
            "source_mask_path": sample["source_mask_path"],
        }


holdout_dataset = HoldoutMulticlassDataset(samples)
holdout_loader = DataLoader(holdout_dataset, batch_size=INFERENCE_BATCH_SIZE, shuffle=False, num_workers=0)

batch = next(iter(holdout_loader))
print("image:", batch["image"].shape, batch["image"].dtype)
print("mask:", batch["mask"].shape, batch["mask"].dtype)
assert batch["image"].shape[1:] == torch.Size([1, TARGET_SIZE[0], TARGET_SIZE[1]])
assert batch["mask"].dtype == torch.long

image: torch.Size([4, 1, 256, 256]) torch.float32
mask: torch.Size([4, 256, 256]) torch.int64


## Metricas y evaluacion holdout

In [13]:
EPS = 1e-8


def multiclass_metrics_from_logits(logits, masks, num_classes=NUM_CLASSES):
    preds = torch.argmax(logits, dim=1)
    dice_by_class = {}
    iou_by_class = {}
    class_presence = {}

    for cls in range(num_classes):
        pred_cls = preds == cls
        mask_cls = masks == cls
        intersection = torch.logical_and(pred_cls, mask_cls).sum().float()
        pred_sum = pred_cls.sum().float()
        mask_sum = mask_cls.sum().float()
        union = torch.logical_or(pred_cls, mask_cls).sum().float()

        if pred_sum.item() == 0 and mask_sum.item() == 0:
            dice = torch.tensor(float("nan"), device=logits.device)
            iou = torch.tensor(float("nan"), device=logits.device)
        else:
            dice = (2.0 * intersection + EPS) / (pred_sum + mask_sum + EPS)
            iou = (intersection + EPS) / (union + EPS)

        dice_by_class[cls] = float(dice.detach().cpu()) if not torch.isnan(dice) else np.nan
        iou_by_class[cls] = float(iou.detach().cpu()) if not torch.isnan(iou) else np.nan
        class_presence[cls] = {
            "gt_present": bool(mask_sum.item() > 0),
            "pred_present": bool(pred_sum.item() > 0),
            "gt_pixels": int(mask_sum.item()),
            "pred_pixels": int(pred_sum.item()),
        }

    no_bg_classes = list(range(1, num_classes))
    dice_macro_no_bg = float(np.nanmean([dice_by_class[c] for c in no_bg_classes]))
    iou_macro_no_bg = float(np.nanmean([iou_by_class[c] for c in no_bg_classes]))
    pixel_accuracy = float((preds == masks).float().mean().detach().cpu())

    return {
        "dice_macro_no_bg": dice_macro_no_bg,
        "iou_macro_no_bg": iou_macro_no_bg,
        "pixel_accuracy": pixel_accuracy,
        "dice_by_class": dice_by_class,
        "iou_by_class": iou_by_class,
        "class_presence": class_presence,
        "preds": preds,
    }

In [14]:
model.eval()
case_metric_rows = []
prediction_cache = []
class_metric_accumulator = {cls: {"dice": [], "iou": [], "gt_absent": 0, "pred_absent": 0} for cls in range(NUM_CLASSES)}

with torch.no_grad():
    for batch in tqdm(holdout_loader, desc="Evaluando holdout"):
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        logits = model(images)

        for i in range(images.shape[0]):
            case_id = batch["case_id"][i]
            single_logits = logits[i:i + 1]
            single_mask = masks[i:i + 1]
            metrics = multiclass_metrics_from_logits(single_logits, single_mask, NUM_CLASSES)
            pred_np = metrics["preds"][0].detach().cpu().numpy().astype(np.int64)
            mask_np = single_mask[0].detach().cpu().numpy().astype(np.int64)
            image_np = images[i, 0].detach().cpu().numpy().astype(np.float32)

            row = {
                "case_id": case_id,
                "slice_index": int(batch["slice_index"][i]),
                "dice_macro_no_bg": metrics["dice_macro_no_bg"],
                "iou_macro_no_bg": metrics["iou_macro_no_bg"],
                "pixel_accuracy": metrics["pixel_accuracy"],
                "source_image_path": batch["source_image_path"][i],
                "source_mask_path": batch["source_mask_path"][i],
            }

            for cls in range(NUM_CLASSES):
                dice_value = metrics["dice_by_class"][cls]
                iou_value = metrics["iou_by_class"][cls]
                row[f"dice_class_{cls}"] = dice_value
                row[f"iou_class_{cls}"] = iou_value
                row[f"gt_class_{cls}_present"] = metrics["class_presence"][cls]["gt_present"]
                row[f"pred_class_{cls}_present"] = metrics["class_presence"][cls]["pred_present"]
                if not np.isnan(dice_value):
                    class_metric_accumulator[cls]["dice"].append(dice_value)
                if not np.isnan(iou_value):
                    class_metric_accumulator[cls]["iou"].append(iou_value)
                if not metrics["class_presence"][cls]["gt_present"]:
                    class_metric_accumulator[cls]["gt_absent"] += 1
                if not metrics["class_presence"][cls]["pred_present"]:
                    class_metric_accumulator[cls]["pred_absent"] += 1

            case_metric_rows.append(row)
            prediction_cache.append({
                "case_id": case_id,
                "image": image_np,
                "mask": mask_np,
                "pred": pred_np,
                "slice_index": int(batch["slice_index"][i]),
            })

metrics_by_case_df = pd.DataFrame(case_metric_rows)
metrics_by_case_csv_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_metrics_by_case.csv"
metrics_by_case_df.to_csv(metrics_by_case_csv_path, index=False)

class_rows = []
for cls in range(NUM_CLASSES):
    dice_values = class_metric_accumulator[cls]["dice"]
    iou_values = class_metric_accumulator[cls]["iou"]
    class_rows.append({
        "class_id": cls,
        "dice_mean": float(np.mean(dice_values)) if dice_values else np.nan,
        "dice_std": float(np.std(dice_values)) if dice_values else np.nan,
        "iou_mean": float(np.mean(iou_values)) if iou_values else np.nan,
        "iou_std": float(np.std(iou_values)) if iou_values else np.nan,
        "n_cases_metric_defined": len(dice_values),
        "n_gt_absent": int(class_metric_accumulator[cls]["gt_absent"]),
        "n_pred_absent": int(class_metric_accumulator[cls]["pred_absent"]),
    })

metrics_by_class_df = pd.DataFrame(class_rows)
metrics_by_class_csv_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_metrics_by_class.csv"
metrics_by_class_df.to_csv(metrics_by_class_csv_path, index=False)

summary_df = pd.DataFrame([{
    "holdout_dice_macro_no_bg": float(metrics_by_case_df["dice_macro_no_bg"].mean()),
    "holdout_iou_macro_no_bg": float(metrics_by_case_df["iou_macro_no_bg"].mean()),
    "holdout_pixel_accuracy": float(metrics_by_case_df["pixel_accuracy"].mean()),
    "n_holdout_cases": int(len(metrics_by_case_df)),
    "slice_strategy_used": slice_strategy_used,
}])
summary_csv_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("Metricas por caso:", metrics_by_case_csv_path)
print("Metricas por clase:", metrics_by_class_csv_path)
print("Summary:", summary_csv_path)
display(summary_df)
display(metrics_by_class_df)

Evaluando holdout:   0%|          | 0/10 [00:00<?, ?it/s]

Metricas por caso: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout/E5_multiclass_holdout_metrics_by_case.csv
Metricas por clase: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout/E5_multiclass_holdout_metrics_by_class.csv
Summary: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout/E5_multiclass_holdout_summary.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,holdout_dice_macro_no_bg,holdout_iou_macro_no_bg,holdout_pixel_accuracy,n_holdout_cases,slice_strategy_used
0,0.83106,0.730884,0.956686,40,center_window_best_prediction


,class_id,dice_mean,dice_std,iou_mean,iou_std,n_cases_metric_defined,n_gt_absent,n_pred_absent
0,0,0.974472,0.023645,0.951183,0.042033,40,0,0
1,1,0.851571,0.134078,0.758233,0.145840,40,0,0
2,2,0.864455,0.155336,0.782012,0.154824,36,4,5
3,3,0.797828,0.115510,0.677065,0.140551,40,0,0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Sanity checks

In [15]:
gt_class_distribution = np.zeros(NUM_CLASSES, dtype=np.float64)
pred_class_distribution = np.zeros(NUM_CLASSES, dtype=np.float64)
masks_out_of_range = []
gt_without_foreground = []
pred_without_foreground = []
cases_class_never_predicted = []

for item in prediction_cache:
    mask = item["mask"]
    pred = item["pred"]
    case_id = item["case_id"]

    if mask.min() < 0 or mask.max() >= NUM_CLASSES:
        masks_out_of_range.append(case_id)
    if not np.any(mask > 0):
        gt_without_foreground.append(case_id)
    if not np.any(pred > 0):
        pred_without_foreground.append(case_id)

    for cls in range(NUM_CLASSES):
        gt_class_distribution[cls] += float(np.mean(mask == cls))
        pred_class_distribution[cls] += float(np.mean(pred == cls))
        if cls > 0 and not np.any(pred == cls):
            cases_class_never_predicted.append({"case_id": case_id, "class_id": cls})

n_cache = max(len(prediction_cache), 1)
gt_class_distribution = gt_class_distribution / n_cache
pred_class_distribution = pred_class_distribution / n_cache

low_dice_classes = [
    int(row["class_id"])
    for _, row in metrics_by_class_df.iterrows()
    if row["class_id"] != 0 and pd.notna(row["dice_mean"]) and float(row["dice_mean"]) < 0.2
]

sanity_checks = {
    "n_cases_evaluated": int(len(prediction_cache)),
    "masks_with_values_out_of_range": masks_out_of_range,
    "cases_gt_without_foreground": gt_without_foreground,
    "cases_pred_without_foreground": pred_without_foreground,
    "cases_where_class_is_never_predicted": cases_class_never_predicted,
    "gt_average_class_distribution": {str(i): float(v) for i, v in enumerate(gt_class_distribution.tolist())},
    "pred_average_class_distribution": {str(i): float(v) for i, v in enumerate(pred_class_distribution.tolist())},
    "classes_with_mean_dice_below_0_2": low_dice_classes,
    "unmapped_original_labels": unmapped_label_events,
}

sanity_checks_json_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_sanity_checks.json"
with open(sanity_checks_json_path, "w", encoding="utf-8") as f:
    json.dump(sanity_checks, f, indent=2, ensure_ascii=False)

print("Sanity checks:", sanity_checks_json_path)
print(json.dumps(sanity_checks, indent=2, ensure_ascii=False)[:4000])

Sanity checks: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout/E5_multiclass_holdout_sanity_checks.json
{
  "n_cases_evaluated": 40,
  "masks_with_values_out_of_range": [],
  "cases_gt_without_foreground": [],
  "cases_pred_without_foreground": [],
  "cases_where_class_is_never_predicted": [
    {
      "case_id": "136_t1",
      "class_id": 2
    },
    {
      "case_id": "18_t1",
      "class_id": 2
    },
    {
      "case_id": "22_t1",
      "class_id": 2
    },
    {
      "case_id": "69_t1",
      "class_id": 2
    },
    {
      "case_id": "78_t1",
      "class_id": 2
    }
  ],
  "gt_average_class_distribution": {
    "0": 0.8469184875488281,
    "1": 0.09814529418945313,
    "2": 0.03273506164550781,
    "3": 0.022201156616210936
  },
  "pred_average_class_distribution": {
    "0": 0.8477333068847657,
    "1": 0.09012489318847657,
    "2": 0.035377120971679686,
    "3": 0.026764678955078124
  },
  "classes_with_mean_dice_below_0_2": [],
  "unmapped_original_labels

## Comparacion contra validacion interna del notebook 09

In [16]:
with open(INTERNAL_REPORT_JSON, "r", encoding="utf-8") as f:
    internal_report = json.load(f)

checkpoint_val_dice_macro_no_bg = multiclass_checkpoint.get("val_dice_macro_no_bg")
report_val_dice_macro_no_bg = internal_report.get("val_dice_macro_no_bg")

comparison_df = pd.DataFrame([
    {
        "metric": "dice_macro_no_bg",
        "internal_validation": report_val_dice_macro_no_bg,
        "holdout": float(summary_df["holdout_dice_macro_no_bg"].iloc[0]),
        "delta_holdout_minus_internal": (
            float(summary_df["holdout_dice_macro_no_bg"].iloc[0]) - float(report_val_dice_macro_no_bg)
            if report_val_dice_macro_no_bg is not None else np.nan
        ),
        "checkpoint_val_dice_macro_no_bg": checkpoint_val_dice_macro_no_bg,
        "report_val_dice_macro_no_bg": report_val_dice_macro_no_bg,
    },
    {
        "metric": "iou_macro_no_bg",
        "internal_validation": internal_report.get("val_iou_macro_no_bg"),
        "holdout": float(summary_df["holdout_iou_macro_no_bg"].iloc[0]),
        "delta_holdout_minus_internal": (
            float(summary_df["holdout_iou_macro_no_bg"].iloc[0]) - float(internal_report.get("val_iou_macro_no_bg"))
            if internal_report.get("val_iou_macro_no_bg") is not None else np.nan
        ),
        "checkpoint_val_dice_macro_no_bg": checkpoint_val_dice_macro_no_bg,
        "report_val_dice_macro_no_bg": report_val_dice_macro_no_bg,
    },
])
comparison_csv_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_vs_internal_validation.csv"
comparison_df.to_csv(comparison_csv_path, index=False)

internal_dice_by_class = {str(k): float(v) for k, v in internal_report.get("dice_by_class", {}).items()}
internal_iou_by_class = {str(k): float(v) for k, v in internal_report.get("iou_by_class", {}).items()}

class_comparison_rows = []
for _, row in metrics_by_class_df.iterrows():
    cls = int(row["class_id"])
    cls_key = str(cls)
    internal_dice = internal_dice_by_class.get(cls_key, np.nan)
    internal_iou = internal_iou_by_class.get(cls_key, np.nan)
    holdout_dice = float(row["dice_mean"]) if pd.notna(row["dice_mean"]) else np.nan
    holdout_iou = float(row["iou_mean"]) if pd.notna(row["iou_mean"]) else np.nan
    class_comparison_rows.append({
        "class_id": cls,
        "internal_dice_mean": internal_dice,
        "holdout_dice_mean": holdout_dice,
        "dice_delta_holdout_minus_internal": holdout_dice - internal_dice if pd.notna(internal_dice) and pd.notna(holdout_dice) else np.nan,
        "internal_iou_mean": internal_iou,
        "holdout_iou_mean": holdout_iou,
        "iou_delta_holdout_minus_internal": holdout_iou - internal_iou if pd.notna(internal_iou) and pd.notna(holdout_iou) else np.nan,
    })

class_comparison_df = pd.DataFrame(class_comparison_rows)
class_comparison_csv_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_class_comparison.csv"
class_comparison_df.to_csv(class_comparison_csv_path, index=False)

print("Comparacion global:", comparison_csv_path)
print("Comparacion por clase:", class_comparison_csv_path)
display(comparison_df)
display(class_comparison_df)

Comparacion global: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout/E5_multiclass_holdout_vs_internal_validation.csv
Comparacion por clase: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout/E5_multiclass_holdout_class_comparison.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,metric,internal_validation,holdout,delta_holdout_minus_internal,checkpoint_val_dice_macro_no_bg,report_val_dice_macro_no_bg
0,dice_macro_no_bg,0.839181,0.831060,-0.008120,0.861357,0.839181
1,iou_macro_no_bg,0.746601,0.730884,-0.015716,0.861357,0.839181


,class_id,internal_dice_mean,holdout_dice_mean,dice_delta_holdout_minus_internal,internal_iou_mean,holdout_iou_mean,iou_delta_holdout_minus_internal
0,0,0.978295,0.974472,-0.003824,0.957686,0.951183,-0.006503
1,1,0.831089,0.851571,0.020482,0.740494,0.758233,0.017739
2,2,0.896068,0.864455,-0.031613,0.818113,0.782012,-0.036101
3,3,0.790386,0.797828,0.007442,0.681195,0.677065,-0.004129


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Visualizaciones

In [17]:
CLASS_COLORS = np.array([
    [0.00, 0.00, 0.00, 0.00],
    [0.90, 0.10, 0.10, 0.55],
    [0.10, 0.65, 0.20, 0.55],
    [0.10, 0.25, 0.95, 0.55],
], dtype=np.float32)


def colorize_mask(mask):
    rgba = np.zeros((*mask.shape, 4), dtype=np.float32)
    for cls in range(NUM_CLASSES):
        rgba[mask == cls] = CLASS_COLORS[cls]
    return rgba


def plot_prediction_example(item, output_path):
    image = item["image"]
    mask = item["mask"]
    pred = item["pred"]
    error_map = (mask != pred).astype(np.float32)

    fig, axes = plt.subplots(1, 6, figsize=(18, 3.4))
    panels = [
        ("Imagen", image, "gray"),
        ("GT agrupada", mask, "viridis"),
        ("Prediccion", pred, "viridis"),
        ("Overlay GT", image, "gray"),
        ("Overlay pred", image, "gray"),
        ("Errores", error_map, "magma"),
    ]
    for ax, (title, data, cmap) in zip(axes, panels):
        ax.imshow(data, cmap=cmap, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")

    axes[3].imshow(colorize_mask(mask), interpolation="nearest")
    axes[4].imshow(colorize_mask(pred), interpolation="nearest")
    fig.suptitle(f"{item['case_id']} - slice {item['slice_index']}")
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


figure_paths = []
example_count = min(3, len(prediction_cache))
ranked_examples = sorted(
    prediction_cache,
    key=lambda item: float(metrics_by_case_df.loc[metrics_by_case_df["case_id"].eq(item["case_id"]), "dice_macro_no_bg"].iloc[0])
)
if example_count >= 3:
    selected_examples = [ranked_examples[0], ranked_examples[len(ranked_examples) // 2], ranked_examples[-1]]
else:
    selected_examples = ranked_examples[:example_count]

for idx, item in enumerate(selected_examples, start=1):
    output_path = FIGURES_ROOT / f"E5_multiclass_holdout_prediction_example_{idx:02d}.png"
    plot_prediction_example(item, output_path)
    figure_paths.append(str(output_path))

dice_by_class_fig_path = FIGURES_ROOT / "E5_multiclass_holdout_dice_by_class.png"
fig, ax = plt.subplots(figsize=(6, 4))
plot_df = metrics_by_class_df.copy()
ax.bar(plot_df["class_id"].astype(str), plot_df["dice_mean"], color=["#7A7A7A", "#D84A4A", "#3AA657", "#3B62D9"])
ax.set_ylim(0, 1)
ax.set_xlabel("Clase")
ax.set_ylabel("Dice medio")
ax.set_title("Holdout - Dice por clase")
fig.tight_layout()
fig.savefig(dice_by_class_fig_path, dpi=160, bbox_inches="tight")
plt.close(fig)
figure_paths.append(str(dice_by_class_fig_path))

print("Figuras exportadas:")
for path in figure_paths:
    print(path)

Figuras exportadas:
/content/drive/MyDrive/PFI_MVP/figures/E5_multiclass_holdout_prediction_example_01.png
/content/drive/MyDrive/PFI_MVP/figures/E5_multiclass_holdout_prediction_example_02.png
/content/drive/MyDrive/PFI_MVP/figures/E5_multiclass_holdout_prediction_example_03.png
/content/drive/MyDrive/PFI_MVP/figures/E5_multiclass_holdout_dice_by_class.png


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Reporte JSON y conclusion tecnica

In [18]:
dice_by_class = {
    str(int(row["class_id"])): (float(row["dice_mean"]) if pd.notna(row["dice_mean"]) else None)
    for _, row in metrics_by_class_df.iterrows()
}
iou_by_class = {
    str(int(row["class_id"])): (float(row["iou_mean"]) if pd.notna(row["iou_mean"]) else None)
    for _, row in metrics_by_class_df.iterrows()
}

recommendation = (
    "El modelo multiclase agrupado muestra una generalizacion holdout razonable si la caida contra validacion interna es moderada "
    "y no hay clases foreground con Dice medio criticamente bajo. El siguiente paso recomendado es revisar casos fallidos, "
    "validar estabilidad de seleccion de slice y luego decidir entre extender evaluacion 2D o preparar baseline 3D/nnU-Net."
)

exports = {
    "selected_cases": str(holdout_selected_csv_path),
    "missing_files": str(holdout_missing_csv_path) if len(missing_files_df) > 0 else None,
    "samples_summary": str(samples_summary_csv_path),
    "unmapped_labels": str(unmapped_labels_csv_path) if unmapped_label_events else None,
    "metrics_by_case": str(metrics_by_case_csv_path),
    "metrics_by_class": str(metrics_by_class_csv_path),
    "summary": str(summary_csv_path),
    "sanity_checks": str(sanity_checks_json_path),
    "holdout_vs_internal": str(comparison_csv_path),
    "class_comparison": str(class_comparison_csv_path),
    "figures": figure_paths,
}

report = {
    "n_t1_candidates": int(len(t1_candidates_df)),
    "n_cases_used_notebook_09": int(len(used_case_ids_09)),
    "n_available_for_holdout": int(len(holdout_pool_df)),
    "n_evaluated": int(len(metrics_by_case_df)),
    "modality": MODALITY_FILTER,
    "num_classes": int(NUM_CLASSES),
    "label_group_mapping": {str(int(k)): int(v) for k, v in LABEL_GROUP_MAPPING.items()},
    "device": str(DEVICE),
    "model_loaded": str(MULTICLASS_MODEL_PATH),
    "binary_selector": binary_selector_status,
    "slice_strategy_used": slice_strategy_used,
    "checkpoint_slice_strategy": CHECKPOINT_SLICE_STRATEGY,
    "holdout_dice_macro_no_bg": float(summary_df["holdout_dice_macro_no_bg"].iloc[0]),
    "holdout_iou_macro_no_bg": float(summary_df["holdout_iou_macro_no_bg"].iloc[0]),
    "holdout_pixel_accuracy": float(summary_df["holdout_pixel_accuracy"].iloc[0]),
    "dice_by_class": dice_by_class,
    "iou_by_class": iou_by_class,
    "comparison_against_internal_validation": comparison_df.to_dict(orient="records"),
    "class_comparison_against_internal_validation": class_comparison_df.to_dict(orient="records"),
    "checkpoint_val_dice_macro_no_bg": checkpoint_val_dice_macro_no_bg,
    "report_val_dice_macro_no_bg": report_val_dice_macro_no_bg,
    "sanity_checks": sanity_checks,
    "exports": exports,
    "recommendation": recommendation,
}

report_json_path = MULTICLASS_HOLDOUT_ROOT / "E5_multiclass_holdout_validation_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Reporte JSON:", report_json_path)

Reporte JSON: /content/drive/MyDrive/PFI_MVP/results/E5_multiclase_holdout/E5_multiclass_holdout_validation_report.json


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
summary_table_md = summary_df.to_markdown(index=False)
class_table_md = metrics_by_class_df.to_markdown(index=False)
comparison_table_md = comparison_df.to_markdown(index=False)
sanity_md = json.dumps(sanity_checks, indent=2, ensure_ascii=False)

conclusion_text = f'''# Conclusion tecnica - E5 validacion holdout multiclase agrupado

## Objetivo

Validar el mejor modelo multiclase agrupado del notebook 09 sobre casos T1 de SPIDER no utilizados en los 100 casos de entrenamiento/validacion interna.

## Por que se hace holdout multiclase

La validacion interna mide desempeno dentro del split usado durante el desarrollo del baseline. Este holdout permite estimar si el mapping agrupado y la U-Net 2D sagital mantienen un desempeno razonable en casos no usados para entrenamiento ni validacion interna.

## Configuracion

- Dataset: SPIDER.
- Modalidad: `{MODALITY_FILTER}`.
- Casos usados en notebook 09 excluidos: {len(used_case_ids_09)}.
- Casos disponibles para holdout: {len(holdout_pool_df)}.
- Casos evaluados: {len(metrics_by_case_df)}.
- Eje sagital: {SAGITTAL_AXIS}.
- Target size: {TARGET_SIZE}.
- Numero de clases: {NUM_CLASSES}.
- Estrategia de slice: `{slice_strategy_used}`.
- Modelo cargado: `{MULTICLASS_MODEL_PATH}`.
- Device: `{DEVICE}`.

## Resultados globales

{summary_table_md}

## Resultados por clase

{class_table_md}

## Comparacion contra validacion interna

{comparison_table_md}

## Sanity checks

```json
{sanity_md}
```

## Interpretacion

La metrica principal es Dice macro sin fondo, acompanada por IoU macro sin fondo. Pixel accuracy se reporta solo como metrica secundaria porque puede estar dominada por el fondo.

La comparacion contra validacion interna debe interpretarse como una prueba de generalizacion dentro del mismo dataset SPIDER. Una caida moderada puede ser esperable; una caida marcada o clases con Dice bajo requieren inspeccion visual y analisis de casos fallidos.

## Limitaciones

- Validacion dentro del mismo dataset SPIDER.
- Mapping agrupado preliminar.
- No se asignan nombres anatomicos definitivos.
- Evaluacion 2D sobre un slice sagital.
- No hay inferencia 3D.
- No se evalua axial todavia.
- No se usa nnU-Net.
- No hay integracion backend.
- El resultado puede depender de la estrategia de seleccion de slice.

## Recomendacion para el proximo paso

{recommendation}

Este notebook no emite diagnosticos, no recomienda tratamientos y no reemplaza revision profesional.
'''

conclusion_path = DOCS_ROOT / "E5_multiclase_holdout_conclusion.md"
with open(conclusion_path, "w", encoding="utf-8") as f:
    f.write(conclusion_text)

print("Conclusion Markdown:", conclusion_path)

Conclusion Markdown: /content/drive/MyDrive/PFI_MVP/docs/E5_multiclase_holdout_conclusion.md


## Criterio de aceptacion

Este notebook cumple el criterio esperado si:

- Excluye los casos usados por el notebook 09.
- Evalua holdout T1 sin reentrenar.
- Carga el mejor modelo multiclase agrupado.
- Usa el mismo mapping agrupado.
- Calcula Dice/IoU macro sin fondo y metricas por clase.
- Exporta CSV, JSON, Markdown y figuras.
- Documenta sanity checks, limitaciones y recomendacion tecnica.